# FIPE AI — Agente Inteligente de Consulta da Tabela FIPE

**Atividade 1 — Agente com ferramenta** · Disciplina de Inteligência Artificial

Este notebook demonstra um **agente de IA com Gemini Function Calling** que consulta preços reais de veículos na Tabela FIPE brasileira através de uma API pública de terceiros, seguindo o padrão *agente + ferramenta*.

## 2. Objetivo

O objetivo desta atividade é demonstrar, na prática, o conceito de **agente com ferramenta** (*tool use* / *function calling*):

- O modelo Gemini **não conhece preços de veículos**: não foi treinado com a Tabela FIPE e não tem acesso a dados atualizados.
- Para responder a perguntas como *"Quanto vale um Toyota Corolla XEi 2022?"*, o Gemini precisa **decidir usar uma ferramenta externa** — uma função Python que faz uma requisição HTTP real a uma API pública da Tabela FIPE.
- Cabe à **IA**: interpretar linguagem natural, identificar a intenção, extrair marca/modelo/ano, decidir quando chamar a ferramenta, fornecer os parâmetros corretos, interpretar o resultado recebido e produzir uma resposta natural.
- Cabe à **ferramenta Python**: realizar a requisição HTTP, consultar a API, obter os dados reais e devolvê-los ao Gemini — sem nenhuma inteligência própria.

Essa separação de responsabilidades é exatamente o que caracteriza um **agente com ferramenta**, e é o que este notebook torna visível, célula a célula.

## 3. Arquitetura

```
Usuário
  │  "Quanto vale um Toyota Corolla XEi 2022?"
  ▼
Gemini  (interpreta a pergunta e decide usar a ferramenta)
  │  function_call: consultar_preco_fipe(marca="Toyota", modelo="Corolla XEi", ano="2022")
  ▼
Python  (executa a função — SEM inteligência, só mecânica de requisição)
  │  GET /cars/brands              → encontra o código da marca
  │  GET /cars/brands/{id}/models  → encontra o código do modelo
  │  GET /cars/.../models/{id}/years        → encontra o código do ano
  │  GET /cars/.../years/{ano}     → preço real
  ▼
API pública da FIPE (Parallelum)
  │  JSON com marca, modelo, ano, preço, mês de referência, código FIPE
  ▼
Python  (devolve o resultado ao Gemini como function_response)
  ▼
Gemini  (interpreta o resultado real e escreve a resposta final)
  ▼
Usuário
  "Encontrei o Toyota Corolla XEi 2022 na Tabela FIPE. O preço médio de
   referência é R$ XX.XXX,XX, na referência de MM/AAAA."
```

Este notebook implementa o fluxo de function calling **manualmente** (em vez do modo automático do SDK) exatamente para deixar cada uma dessas etapas visível nas células e nas saídas impressas abaixo — a intenção é que seja fácil identificar onde está o agente e onde está a ferramenta.

## 4. Instalação

Apenas duas bibliotecas de terceiros são necessárias:

- **`google-genai`** — SDK oficial do Google para a Gemini API (cliente + function calling).
- **`requests`** — para as chamadas HTTP GET à API pública da FIPE.

Nada além disso é usado neste notebook.

In [ ]:
%pip install -q google-genai requests

## 5. Imports

In [ ]:
import difflib
import unicodedata
from functools import lru_cache
from typing import Any

import requests
from google import genai
from google.genai import types
from google.colab import userdata

print("Bibliotecas importadas com sucesso.")

## 6. Configuração segura da API Gemini

A chave da API do Gemini **nunca** é escrita no código. Ela é lida a partir dos **Secrets do Google Colab**:

1. No menu lateral do Colab, clique no ícone de chave (🔑) **Secrets**.
2. Crie um secret chamado `GEMINI_API_KEY` com o valor da sua chave (gerada em [aistudio.google.com/apikey](https://aistudio.google.com/apikey)).
3. Habilite o acesso deste notebook a esse secret (toggle "Notebook access").

A célula abaixo apenas **lê** o secret pelo nome — a chave não aparece em nenhum momento no código-fonte nem é impressa em nenhuma saída.

In [ ]:
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Não foi possível ler o secret 'GEMINI_API_KEY'. Configure-o no menu Secrets (chave, "
        "na barra lateral esquerda) e habilite o acesso deste notebook a ele."
    ) from exc

if not GEMINI_API_KEY:
    raise RuntimeError("O secret 'GEMINI_API_KEY' está vazio. Preencha-o em Secrets antes de continuar.")

# Modelo Gemini com suporte a function calling. Modelos são descontinuados com
# o tempo (isso aconteceu durante o desenvolvimento deste notebook, quando o
# gemini-2.5-flash deixou de estar disponível para novos usuários) — se a
# célula abaixo falhar informando que o modelo não existe mais, troque o valor
# pelo modelo recomendado na própria mensagem de erro do Gemini.
GEMINI_MODEL = "gemini-3.6-flash"

client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Cliente Gemini configurado. Modelo: {GEMINI_MODEL}")

## 7. Configuração da API FIPE

Usamos a **Fipe API v2**, mantida por Deivid Fortuna (documentação em <https://deividfortuna.github.io>), hospedada em:

```
https://fipe.parallelum.com.br/api/v2
```

É uma API pública e gratuita: permite **500 requisições/dia por IP sem autenticação**. Opcionalmente é possível criar um token gratuito em [fipe.api.br](https://fipe.api.br) para elevar esse limite a 1.000/dia — por isso há suporte a um token opcional via secret `FIPE_API_TOKEN`, mas ele **não é obrigatório** para este notebook funcionar.

A hierarquia de navegação confirmada na documentação é **tipo de veículo → marca → modelo → ano → preço**. Não existe endpoint de busca livre por nome — por isso a ferramenta da Seção 9 primeiro busca as listas de marcas/modelos/anos e faz a correspondência com o texto informado pelo usuário.

In [ ]:
FIPE_BASE_URL = "https://fipe.parallelum.com.br/api/v2"
FIPE_TIMEOUT_SEGUNDOS = 10

# Opcional: só é enviado se você definir o secret FIPE_API_TOKEN no Colab.
try:
    FIPE_API_TOKEN = userdata.get("FIPE_API_TOKEN")
except Exception:
    FIPE_API_TOKEN = None

TIPOS_VEICULO_VALIDOS = {
    "carro": "cars", "carros": "cars", "automovel": "cars", "automóvel": "cars",
    "moto": "motorcycles", "motos": "motorcycles", "motocicleta": "motorcycles", "motocicletas": "motorcycles",
    "caminhao": "trucks", "caminhão": "trucks", "caminhoes": "trucks", "caminhões": "trucks",
}

print(f"API FIPE configurada em: {FIPE_BASE_URL}")
print("Token opcional definido." if FIPE_API_TOKEN else "Sem token — usando o limite gratuito de 500 requisições/dia.")

## 8. Funções auxiliares

Estas funções **não são expostas ao Gemini** — são utilitários internos que a ferramenta principal (Seção 9) usa para navegar a hierarquia da API: normalizar texto, chamar a API com tratamento de erros, buscar a marca mais próxima do que o usuário digitou, rankear modelos candidatos e localizar o código de ano correto.

O cache (`@lru_cache`) evita repetir chamadas HTTP para a mesma marca/modelo dentro da sessão, para não sobrecarregar a API pública.

In [ ]:
def _normalizar(texto: str) -> str:
    """Remove acentos, baixa a caixa e colapsa espaços para comparação de texto."""
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return " ".join(sem_acento.lower().split())


class FipeApiError(Exception):
    """Erro ao consultar a API pública da FIPE (rede, HTTP ou dados inesperados)."""

    def __init__(self, tipo: str, mensagem: str):
        self.tipo = tipo
        self.mensagem = mensagem
        super().__init__(mensagem)


def _get_json(path: str, params: dict[str, Any] | None = None) -> Any:
    """Executa um GET na API da FIPE e retorna o JSON, ou levanta FipeApiError."""
    headers = {"Accept": "application/json"}
    if FIPE_API_TOKEN:
        headers["X-Subscription-Token"] = FIPE_API_TOKEN

    try:
        resposta = requests.get(f"{FIPE_BASE_URL}{path}", params=params, headers=headers, timeout=FIPE_TIMEOUT_SEGUNDOS)
    except requests.exceptions.Timeout as exc:
        raise FipeApiError("timeout", "A API da FIPE não respondeu a tempo. Tente novamente em instantes.") from exc
    except requests.exceptions.ConnectionError as exc:
        raise FipeApiError("conexao", "Não foi possível conectar à API da FIPE. Verifique sua conexão.") from exc
    except requests.exceptions.RequestException as exc:
        raise FipeApiError("requisicao", f"Falha ao consultar a API da FIPE: {exc}") from exc

    if resposta.status_code == 200:
        try:
            return resposta.json()
        except ValueError as exc:
            raise FipeApiError("json_invalido", "A API da FIPE retornou uma resposta em formato inesperado.") from exc

    if resposta.status_code == 404:
        raise FipeApiError("nao_encontrado", "Recurso não encontrado na Tabela FIPE para os parâmetros informados.")
    if resposta.status_code == 429:
        raise FipeApiError("limite_excedido", "Limite de requisições da API FIPE foi atingido. Tente novamente mais tarde.")
    if resposta.status_code >= 500:
        raise FipeApiError("servico_indisponivel", "A API da FIPE está indisponível no momento.")

    raise FipeApiError("http_" + str(resposta.status_code), f"A API da FIPE retornou o status HTTP {resposta.status_code}.")


print("Camada HTTP definida: _normalizar, FipeApiError, _get_json")

In [ ]:
@lru_cache(maxsize=8)
def _listar_marcas(tipo_api: str) -> tuple[dict[str, str], ...]:
    """Lista marcas de um tipo de veículo. Cacheada: marcas mudam raramente."""
    return tuple(_get_json(f"/{tipo_api}/brands"))


@lru_cache(maxsize=64)
def _listar_modelos(tipo_api: str, marca_codigo: str) -> tuple[dict[str, str], ...]:
    """Lista modelos de uma marca. Cacheada para evitar chamadas repetidas."""
    return tuple(_get_json(f"/{tipo_api}/brands/{marca_codigo}/models"))


@lru_cache(maxsize=256)
def _listar_anos(tipo_api: str, marca_codigo: str, modelo_codigo: str) -> tuple[dict[str, str], ...]:
    """Lista anos/versões de combustível de um modelo. Cacheada."""
    return tuple(_get_json(f"/{tipo_api}/brands/{marca_codigo}/models/{modelo_codigo}/years"))


def _melhor_marca(tipo_api: str, marca_usuario: str) -> tuple[dict[str, str] | None, tuple[dict[str, str], ...]]:
    """Encontra a marca cujo nome mais se aproxima do texto informado pelo usuário."""
    marcas = _listar_marcas(tipo_api)
    alvo = _normalizar(marca_usuario)

    exatas = [m for m in marcas if _normalizar(m["name"]) == alvo]
    if exatas:
        return exatas[0], marcas

    contendo = [m for m in marcas if alvo in _normalizar(m["name"])]
    if contendo:
        return contendo[0], marcas

    nomes_normalizados = {_normalizar(m["name"]): m for m in marcas}
    proximos = difflib.get_close_matches(alvo, nomes_normalizados.keys(), n=1, cutoff=0.6)
    if proximos:
        return nomes_normalizados[proximos[0]], marcas

    return None, marcas


print("Funções de busca de marca definidas: _listar_marcas, _listar_modelos, _listar_anos, _melhor_marca")

In [ ]:
def _rankear_modelos(modelos, modelo_usuario: str):
    """
    Separa os modelos em três níveis de confiança: correspondência exata,
    correspondência por subconjunto de palavras e correspondência difusa (fuzzy).

    Os nomes de modelo na FIPE incluem detalhes de versão/motorização (ex.:
    "Corolla XEi 2.0 Flex 16V Aut."), então uma busca exata quase nunca bate.
    Importante: dentro de cada nível NÃO se penaliza nomes mais longos/detalhados
    — testamos isso na prática: penalizar nomes longos faz trims antigos e
    genéricos (ex.: "Corolla WG", vendido até 1999) vencerem versões atuais
    sempre que o usuário não especifica a versão, o que é o oposto do desejável.
    """
    alvo = _normalizar(modelo_usuario)
    alvo_tokens = set(alvo.split())
    exatos, subconjunto, difusos = [], [], []

    for modelo in modelos:
        nome_normalizado = _normalizar(modelo["name"])
        if nome_normalizado == alvo:
            exatos.append(modelo)
            continue

        tokens_modelo = set(nome_normalizado.split())
        if alvo_tokens and alvo_tokens.issubset(tokens_modelo):
            subconjunto.append(modelo)
            continue

        similaridade = difflib.SequenceMatcher(None, alvo, nome_normalizado).ratio()
        if alvo in nome_normalizado or similaridade > 0.5:
            difusos.append((similaridade, modelo))

    difusos.sort(key=lambda item: item[0], reverse=True)
    return exatos, subconjunto, [modelo for _, modelo in difusos]


def _ano_para_codigo(anos, ano_usuario: str):
    """Encontra o código de ano (ex.: '2022-5') correspondente ao ano informado (ex.: '2022')."""
    alvo = str(ano_usuario).strip()[:4]
    for ano in anos:
        if ano["code"].split("-")[0] == alvo:
            return ano
    return None


def _anos_disponiveis_legiveis(anos, limite: int = 10):
    """Lista anos reais disponíveis (ignorando o placeholder '32000' da FIPE) para mensagens de erro."""
    anos_unicos = sorted({a["code"].split("-")[0] for a in anos if a["code"].split("-")[0] != "32000"}, reverse=True)
    return anos_unicos[:limite]


print("Funções de busca de modelo/ano definidas: _rankear_modelos, _ano_para_codigo, _anos_disponiveis_legiveis")

## 9. Ferramenta principal: `consultar_preco_fipe`

Esta é a **única função exposta ao Gemini como ferramenta**. Ela recebe os parâmetros que o próprio Gemini já extraiu da pergunta do usuário (marca, modelo, ano e, opcionalmente, o tipo de veículo) e devolve um dicionário com o resultado real da consulta — nunca um valor inventado.

Internamente ela executa a navegação hierárquica completa (marca → modelo → ano → preço), tentando algumas variantes de modelo quando o nome exato informado não tem o ano pedido. Isso não é hipotético: testamos com a API real durante o desenvolvimento e confirmamos que uma "Honda CG 160 Titan 2023" só existe na FIPE sob o nome de uma edição especial daquele ano ("CG 160 TITAN FLEXONE/Ed.Especial 40 Anos") — o modelo "CG 160 TITAN" puro só tem dados de 2026. Por isso a função tenta algumas variantes antes de desistir, e retorna um erro claro (com sugestões reais) quando realmente não encontra nada, em vez de travar ou inventar um valor.

In [ ]:
# Se a correspondência por subconjunto de palavras encontrar mais candidatos do
# que isso, o nome do modelo é considerado genérico demais (ex.: "Corolla" sem
# versão) e a função retorna um erro de ambiguidade em vez de arriscar checar
# o ano em dezenas de trims diferentes (o que também evita excesso de chamadas
# à API pública).
LIMITE_MODELOS_AMBIGUOS = 6

# Número máximo de modelos candidatos testados contra o ano pedido.
LIMITE_TENTATIVAS_DE_ANO = 5


def consultar_preco_fipe(marca: str, modelo: str, ano: str, tipo_veiculo: str = "carros") -> dict[str, Any]:
    """
    Consulta o preço médio de um veículo na Tabela FIPE através da API pública.

    Args:
        marca: Marca/fabricante do veículo (ex.: "Toyota", "Honda").
        modelo: Modelo e, se possível, a versão do veículo (ex.: "Corolla XEi").
        ano: Ano-modelo do veículo (ex.: "2022").
        tipo_veiculo: "carros", "motos" ou "caminhoes". Padrão: "carros".

    Returns:
        Em caso de sucesso: {"sucesso": True, "marca", "modelo", "ano_modelo",
        "combustivel", "preco", "codigo_fipe", "mes_referencia", "tipo_veiculo"}.
        Em caso de falha: {"sucesso": False, "erro": <categoria>, "mensagem": <texto>,
        e opcionalmente "sugestoes" ou "anos_disponiveis"}.
    """
    if not marca or not str(marca).strip():
        return {"sucesso": False, "erro": "parametros_invalidos", "mensagem": "É necessário informar a marca do veículo."}
    if not modelo or not str(modelo).strip():
        return {"sucesso": False, "erro": "parametros_invalidos", "mensagem": "É necessário informar o modelo do veículo."}
    if not ano or not str(ano).strip():
        return {"sucesso": False, "erro": "parametros_invalidos", "mensagem": "É necessário informar o ano do veículo."}

    tipo_api = TIPOS_VEICULO_VALIDOS.get(_normalizar(tipo_veiculo))
    if tipo_api is None:
        return {
            "sucesso": False,
            "erro": "tipo_invalido",
            "mensagem": f"Tipo de veículo '{tipo_veiculo}' não reconhecido. Utilize carros, motos ou caminhões.",
        }

    try:
        marca_info, marcas = _melhor_marca(tipo_api, marca)
        if marca_info is None:
            return {
                "sucesso": False,
                "erro": "marca_nao_encontrada",
                "mensagem": f"Não encontrei a marca '{marca}' na Tabela FIPE.",
                "sugestoes": [m["name"] for m in marcas[:5]],
            }

        modelos = _listar_modelos(tipo_api, marca_info["code"])
        exatos, subconjunto, difusos = _rankear_modelos(modelos, modelo)

        if not exatos and len(subconjunto) > LIMITE_MODELOS_AMBIGUOS:
            return {
                "sucesso": False,
                "erro": "modelo_ambiguo",
                "mensagem": (
                    f"Encontrei {len(subconjunto)} versões diferentes de '{modelo}' para a marca "
                    f"{marca_info['name']}. Preciso que você especifique a versão exata."
                ),
                "sugestoes": [m["name"] for m in subconjunto[:8]],
            }

        candidatos = exatos + subconjunto[:LIMITE_TENTATIVAS_DE_ANO]
        if not candidatos:
            candidatos = difusos[:LIMITE_TENTATIVAS_DE_ANO]
        if not candidatos:
            return {
                "sucesso": False,
                "erro": "modelo_nao_encontrado",
                "mensagem": f"Não encontrei o modelo '{modelo}' para a marca {marca_info['name']}.",
                "sugestoes": [m["name"] for m in modelos[:5]],
            }

        for candidato in candidatos[:LIMITE_TENTATIVAS_DE_ANO]:
            anos = _listar_anos(tipo_api, marca_info["code"], candidato["code"])
            ano_info = _ano_para_codigo(anos, ano)
            if ano_info is None:
                continue

            dados = _get_json(f"/{tipo_api}/brands/{marca_info['code']}/models/{candidato['code']}/years/{ano_info['code']}")
            return {
                "sucesso": True,
                "marca": dados.get("brand"),
                "modelo": dados.get("model"),
                "ano_modelo": dados.get("modelYear"),
                "combustivel": dados.get("fuel"),
                "preco": dados.get("price"),
                "codigo_fipe": dados.get("codeFipe"),
                "mes_referencia": dados.get("referenceMonth"),
                "tipo_veiculo": tipo_veiculo,
            }

        anos_do_melhor_candidato = _listar_anos(tipo_api, marca_info["code"], candidatos[0]["code"])
        return {
            "sucesso": False,
            "erro": "ano_nao_encontrado",
            "mensagem": (
                f"Encontrei o modelo '{candidatos[0]['name']}' da marca {marca_info['name']}, "
                f"mas não há dados na FIPE para o ano {ano}."
            ),
            "anos_disponiveis": _anos_disponiveis_legiveis(anos_do_melhor_candidato),
        }

    except FipeApiError as exc:
        return {"sucesso": False, "erro": exc.tipo, "mensagem": exc.mensagem}


print("Ferramenta consultar_preco_fipe definida.")

## 10. Declaração da ferramenta para o Gemini

Para que o Gemini saiba que essa ferramenta existe e quando usá-la, declaramos seu **nome**, uma **descrição clara** do que ela faz e o **schema dos parâmetros** (nome, tipo, descrição e quais são obrigatórios).

O Gemini usa exatamente essas descrições para decidir **se** deve chamar a função e **quais argumentos** extrair da pergunta do usuário — por isso a clareza da descrição importa tanto quanto o código da própria função.

In [ ]:
CONSULTAR_PRECO_FIPE_DECLARATION = types.FunctionDeclaration(
    name="consultar_preco_fipe",
    description=(
        "Consulta o preço médio de referência de um veículo na Tabela FIPE através de uma "
        "API pública oficial de dados da FIPE. Use sempre que o usuário perguntar quanto "
        "vale, qual o preço, cotação ou valor de revenda de um veículo específico. É "
        "necessário saber marca, modelo e ano; se alguma dessas informações não estiver "
        "clara na pergunta do usuário, peça esclarecimento ANTES de chamar esta função."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "marca": types.Schema(
                type=types.Type.STRING,
                description="Marca/fabricante do veículo, por exemplo: Toyota, Honda, Volkswagen.",
            ),
            "modelo": types.Schema(
                type=types.Type.STRING,
                description=(
                    "Modelo do veículo, incluindo a versão/trim quando o usuário mencionar "
                    "(ex.: 'Corolla XEi', 'CG 160 Titan', 'Gol 1.0')."
                ),
            ),
            "ano": types.Schema(
                type=types.Type.STRING,
                description="Ano-modelo do veículo, por exemplo: '2022'.",
            ),
            "tipo_veiculo": types.Schema(
                type=types.Type.STRING,
                description="Categoria do veículo. Use 'carros' quando não for possível inferir outra categoria.",
                enum=["carros", "motos", "caminhoes"],
            ),
        },
        required=["marca", "modelo", "ano"],
    ),
)

FIPE_TOOL = types.Tool(function_declarations=[CONSULTAR_PRECO_FIPE_DECLARATION])

print("Ferramenta declarada para o Gemini:", CONSULTAR_PRECO_FIPE_DECLARATION.name)

## 11. Agente

Configuramos o comportamento do agente com uma **instrução de sistema** (system instruction) e registramos a ferramenta através de `tools=[...]`. A instrução de sistema reforça as regras que a atividade exige: nunca inventar preços, pedir esclarecimento em caso de pergunta incompleta ou ambígua, e sempre basear a resposta final nos dados reais devolvidos pela ferramenta.

In [ ]:
SYSTEM_INSTRUCTION = """Você é o FIPE AI, um assistente que ajuda pessoas a consultar preços de veículos na Tabela FIPE brasileira.

Regras importantes:
1. Você nunca sabe o preço de um veículo de antemão. A única forma de descobrir é chamando a função consultar_preco_fipe.
2. Se o usuário não informar claramente marca, modelo e ano, peça essas informações antes de chamar a função. Não presuma valores.
3. Depois de receber o resultado da função, use somente os dados retornados para responder. Nunca invente ou arredonde valores.
4. Se a função retornar um erro (marca/modelo/ano não encontrado, ambiguidade entre versões, etc.), explique o problema de forma clara e amigável, aproveitando as sugestões retornadas para orientar o usuário a reformular a pergunta.
5. Sempre cite o mês/ano de referência da tabela na resposta final, quando disponível.
6. Seja objetivo e cordial, respondendo sempre em português do Brasil.
"""

GEMINI_CONFIG = types.GenerateContentConfig(
    tools=[FIPE_TOOL],
    system_instruction=SYSTEM_INSTRUCTION,
)

FUNCOES_DISPONIVEIS = {"consultar_preco_fipe": consultar_preco_fipe}

print("Agente configurado com a ferramenta consultar_preco_fipe e instrução de sistema.")

## 12. Fluxo de Function Calling

A função abaixo implementa o ciclo **manual** de function calling do Gemini — de propósito, para que cada etapa exigida pela atividade fique visível na saída:

1. Envia a pergunta do usuário ao Gemini.
2. Se o Gemini decidir usar a ferramenta, a resposta contém uma `function_call` (nome + argumentos) em vez de texto direto.
3. Executamos a função Python correspondente localmente — é aqui que a API da FIPE é chamada de verdade.
4. Devolvemos o resultado ao Gemini como `function_response`.
5. Pedimos uma nova geração — agora o Gemini já tem o resultado real e escreve a resposta final.

Se o Gemini perceber que faltam informações (ex.: ano não informado), ele responde diretamente com uma pergunta de esclarecimento, **sem** chamar a ferramenta — esse caso é tratado no passo "sem function_call" abaixo, e é demonstrado na Seção 15.

In [ ]:
def perguntar_ao_agente(pergunta: str) -> str:
    print(f"Usuário: {pergunta}")
    print("-" * 70)

    contents = [types.Content(role="user", parts=[types.Part(text=pergunta)])]

    # 1) Pergunta do usuário enviada ao Gemini junto com a ferramenta disponível.
    resposta = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=GEMINI_CONFIG)
    candidate_content = resposta.candidates[0].content
    partes = candidate_content.parts or []
    function_call_part = next((p for p in partes if p.function_call), None)

    # 2) Se não há function_call, o Gemini respondeu direto (ex.: pediu esclarecimento).
    if function_call_part is None:
        print(f"FIPE AI: {resposta.text}")
        return resposta.text

    function_call = function_call_part.function_call
    nome_funcao = function_call.name
    argumentos = dict(function_call.args or {})

    # 3) Decisão do Gemini de usar a ferramenta, com nome e argumentos extraídos.
    print(f"[Gemini decidiu usar a ferramenta: {nome_funcao}]")
    print(f"[Argumentos extraídos da pergunta: {argumentos}]")

    contents.append(candidate_content)

    # 4) Execução real da função Python (que faz o HTTP GET na API da FIPE).
    funcao_python = FUNCOES_DISPONIVEIS[nome_funcao]
    resultado = funcao_python(**argumentos)

    # 5) Resultado real vindo da API, antes de qualquer interpretação do Gemini.
    print(f"[Resultado da ferramenta / API FIPE]: {resultado}")
    print("-" * 70)

    # 6) O resultado volta para o Gemini como function_response.
    function_response_part = types.Part.from_function_response(name=nome_funcao, response={"result": resultado})
    contents.append(types.Content(role="user", parts=[function_response_part]))

    # 7) Resposta final do Gemini, já interpretando o resultado real.
    resposta_final = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=GEMINI_CONFIG)
    print(f"FIPE AI: {resposta_final.text}")
    return resposta_final.text

## 13. Interação 1 — consulta direta de um carro

Marca, modelo (com versão) e ano informados claramente na pergunta.

In [ ]:
_ = perguntar_ao_agente("Qual é o preço FIPE de um Toyota Corolla XEi 2022?")

## 14. Interação 2 — outro veículo, outra categoria (moto)

Mesmo fluxo, agora para a categoria "motos" — o Gemini precisa perceber que se trata de uma motocicleta e passar `tipo_veiculo="motos"`.

In [ ]:
_ = perguntar_ao_agente("Quanto vale uma Honda CG 160 Titan 2023?")

## 15. Exemplo de erro e ambiguidade

Duas situações que a atividade exige que sejam tratadas sem que a aplicação quebre ou invente dados:

**(a) Pergunta incompleta.** O usuário não informa o ano nem a versão. O ideal é que o Gemini perceba que falta informação obrigatória e pergunte, em vez de chamar a ferramenta com dados incompletos ou adivinhados. (O comportamento exato pode variar entre execuções, por depender do modelo de linguagem — mas a instrução de sistema da Seção 11 e o parâmetro `ano` marcado como obrigatório na Seção 10 orientam fortemente o Gemini a pedir esclarecimento.)

In [ ]:
_ = perguntar_ao_agente("Quanto vale um Corolla?")

**(b) Veículo/ano inexistente na base.** Aqui a marca, o modelo e o ano estão claros — o Gemini chamará a ferramenta normalmente — mas o ano pedido não existe para esse modelo na Tabela FIPE. Isso exercita o tratamento de erro `ano_nao_encontrado` da ferramenta (Seção 9), com sugestões de anos reais que existem, sem inventar nenhum valor.

In [ ]:
_ = perguntar_ao_agente("Qual o preço FIPE de um Toyota Corolla XEi de 1990?")

## 16. Conclusão

Este notebook demonstrou, de ponta a ponta, um **agente de IA com ferramenta (tool use)**:

- O **Gemini** foi responsável por interpretar linguagem natural, decidir quando a ferramenta `consultar_preco_fipe` era necessária, extrair marca/modelo/ano da pergunta, pedir esclarecimento quando a pergunta era ambígua ou incompleta, e transformar o resultado técnico da API em uma resposta natural.
- A **ferramenta Python** não teve nenhuma inteligência própria: apenas navegou a hierarquia da API pública da FIPE (tipo → marca → modelo → ano → preço) via requisições HTTP GET reais, tratou erros (marca/modelo/ano não encontrados, ambiguidade, timeout, indisponibilidade) e devolveu dados reais — nunca inventados — ao Gemini.
- A chave da API do Gemini nunca foi exposta no código-fonte, sendo lida exclusivamente dos **Secrets do Google Colab**.

Esse padrão — modelo de linguagem + função externa registrada como ferramenta + API real — é a base de agentes de IA modernos capazes de agir sobre o mundo real, e não apenas gerar texto a partir do que já sabem.